# SolarLoss — protected shading evaluation v2

Retrospective, dataset-aware experiment on checksum-verified La Réunion measurements. The historical notebook remains unchanged. This notebook executes E1–E4 and displays E5 as a source-backed proposal. No causal repair recovery, commercial savings or universal fault diagnosis is inferred.

CPU workflow: ~1.8 GB source cache; weather reads in 500,000-row chunks. Electrical frames retain compact columns; allow several GB RAM. Use Drive caching to avoid repeated downloads. Upload the complete analysis ZIP if the private repository is unavailable in Colab. Run all cells in order. The frozen protocol is read before any protected evaluation. Source files: [Zenodo](https://zenodo.org/records/7291380), [data paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC9800176/), [setup paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC9497466/).

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
configured = os.environ.get('SOLARLOSS_ANALYSIS_DIR')
candidates = [Path(configured)] if configured else []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base/'analysis', base])
ANALYSIS_DIR = next((p.resolve() for p in candidates if (p/'EXPERIMENT_PROTOCOL_V2.md').exists()), None)
if ANALYSIS_DIR is None:
    from google.colab import files
    uploaded = files.upload()  # Select the complete SolarLoss v2 integration ZIP.
    bundle = next(Path(name) for name in uploaded if name.endswith('.zip'))
    dest = Path('/content/solarloss_v2'); dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(bundle) as z:
        for item in z.infolist():
            target=(dest/item.filename).resolve()
            if dest.resolve() not in target.parents and target!=dest.resolve():
                raise ValueError('Unsafe ZIP path')
        z.extractall(dest)
    ANALYSIS_DIR=dest/'analysis'
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ANALYSIS_DIR/'requirements-v2.txt')])
sys.path.insert(0,str(ANALYSIS_DIR/'src'))
print('Analysis:',ANALYSIS_DIR)

## Cache and frozen protocol

Drive mounting applies only in Colab. Local execution uses the same real-byte cache through environment variables. `INCLUDE_WEATHER=False` is an explicitly different fallback experiment: its model choice and metrics must not be relabelled as the weather-enabled result.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_root=Path('/content/drive/MyDrive/solarloss_phase2')
except ImportError:
    cache_root=ANALYSIS_DIR/'data'
DATA_DIR=Path(os.environ.get('SOLARLOSS_DATA_DIR',str(cache_root/'raw')))
RESULTS_DIR=Path(os.environ.get('SOLARLOSS_RESULTS_DIR',str(ANALYSIS_DIR/'results')))
INCLUDE_WEATHER=True
from IPython.display import display, Markdown, Image
display(Markdown((ANALYSIS_DIR/'EXPERIMENT_PROTOCOL_V2.md').read_text(encoding='utf-8')))
print('Source cache:', DATA_DIR)

## Execute the complete evaluation

Downloads are streamed and published sizes/MD5s are verified before processing; SHA256 is also recorded. Whole pre-event local days define separate fitting, model-selection and threshold-calibration populations. Frozen parameters are written before protected scoring. No weather transformation is fitted on later data.

In [ ]:
from solarloss_research.evaluation_v2 import run
result=run(DATA_DIR,RESULTS_DIR,ANALYSIS_DIR/'EXPERIMENT_PROTOCOL_V2.md',include_weather=INCLUDE_WEATHER)
print('Finished:',result['verdicts'])

## Verify processing and scientific mechanics

In [ ]:
subprocess.check_call([sys.executable,'-m','pytest',str(ANALYSIS_DIR/'tests'),'-q'])
import json, pandas as pd
audit=json.loads((RESULTS_DIR/'data_quality_report_v2.json').read_text())
frozen=json.loads((RESULTS_DIR/'frozen_parameters_v2.json').read_text())
comparison=json.loads((RESULTS_DIR/'model_comparison_v2.json').read_text())
display(pd.DataFrame(audit['checks'])[['file','bytes','md5','sha256','checksum_matches']])
display(pd.DataFrame({k:{'rows':v['row_count'],'first':v['timestamp_start_utc'],'last':v['timestamp_end_utc'],'median_s':v['interval_seconds']['median'],'duplicates':v['duplicate_timestamp_count']} for k,v in audit['audits'].items()}).T)
display(pd.DataFrame(frozen['populations']).T)
print('Alignment:',audit['alignment'])
print('Protocol SHA256:',result['protocol_sha256'])

## Train, select, calibrate, then evaluate

In [ ]:
display(Image(filename=str(RESULTS_DIR/'charts/timeline.png')))
display(pd.DataFrame(comparison['selection_scores']).T)
display(pd.DataFrame(comparison['comparisons']).T)
display(pd.DataFrame(comparison['common_weather_later_scores']).T)
display(Image(filename=str(RESULTS_DIR/'charts/model_comparison.png')))
print('Selected only on early healthy data:',result['selected_model'])
print('Later healthy stability:',result['reference_stability'])

## Automatic detector and physical labels

The detector runs over the full stream with predictor-only availability screens. Fault is read afterwards for evaluation. Recall is by grouped physical episode; healthy false alarms are alert runs per evaluated healthy hour. Window confusion and precision are separate statistics for this labelled dataset, not commercial predictive values.

In [ ]:
events=pd.read_csv(RESULTS_DIR/'episode_metrics_v2.csv')
display(events[['event_id','start','labels','detected','latency_seconds','coverage']])
display(pd.DataFrame(result['healthy_stability']).T)
display(pd.DataFrame({k:{'detected':v['detected_episodes'],'episodes':v['episode_count'],'healthy_alarm_rate':v['healthy_stability']['false_alarm_runs_per_healthy_hour']} for k,v in result['baseline_detection'].items()}).T)
print('Window confusion:',result['window_confusion'])
print('Sensitivity:',result['window_sensitivity'])
print('Missed:',result['missed_event_ids'])
print('Recall interval (exploratory intervention-day bootstrap):',result['recall_intervention_day_bootstrap_95pct'])
display(Image(filename=str(RESULTS_DIR/'charts/demo_episode.png')))

## Signed energy and model-error comparisons

Integrate only consecutive eligible original records using actual elapsed seconds and gaps <=18s. Signed and positive-only energy are distinct. Positive-only energy cannot establish abnormality. Empirical placebo bands do not identify true unshaded target generation or repair benefits.

In [ ]:
examples=json.loads((RESULTS_DIR/'example_cases_v2.json').read_text())
display(events[['event_id','valid_seconds','coverage','actual_energy_kwh','expected_energy_kwh','signed_energy_kwh','positive_only_kwh']])
display(Image(filename=str(RESULTS_DIR/'charts/event_energy.png')))
display(pd.DataFrame([{ 'event_id':e['event_id'], **e['placebo']} for e in result['episodes']]))
print('Healthy example:',examples['healthy_interval'])
print('Insufficient-data example:',examples['insufficient_data'])
print('Demo event:',result['demo_event_id'])

## Electrical observations and proposed guidance

Operating Va is PV-side voltage, not grid voltage or Voc. Ia is PV current; Ig and Pg are related AC measurements, so Ig is not independent confirmation of power underperformance. Envelopes are trained before shading; matched comparisons are descriptive and remain confounded by operating points and season.

In [ ]:
display(Image(filename=str(RESULTS_DIR/'charts/electrical_observations.png')))
display(pd.DataFrame([{'event_id':e['event_id'],'signal':s,**v} for e in result['episodes'] for s,v in e['electrical'].items()]))
for name in ['ELECTRICAL_EVIDENCE_MATRIX.md','INVESTIGATION_GUIDANCE_PROPOSAL.md','SCHEMA_FEEDBACK_V2.md']:
    display(Markdown((ANALYSIS_DIR/name).read_text(encoding='utf-8')))
print('Research files written:', RESULTS_DIR)
print('Limitations:',result['limitations'])

## Reproducibility and integration

Research JSONs, CSV numbers and external charts are reusable. No production v0.1 review artifact is created. Rachit must approve metric semantics and proposed v0.2 contract before backend/frontend integration. Preserve the historical v1 results. See README_V2.md and the executive report for exact populations, verdicts, failure modes and demonstration claims.